In [5]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from mamba_model import Mamba,MyCustomMamba, MambaConfig
import argparse
from DataProcessing import HankelMatrices
import scipy.io
import l4casadi as l4c
import casadi as cs

In [6]:
# Load multisine data from matlab
mat = scipy.io.loadmat("DataSets/u_dataChirpWhiteNoise.mat")
u_data = mat["u_data"]

mat = scipy.io.loadmat("DataSets/y_dataChirpWhiteNoise.mat")
y_data = mat["y_data"]

# convert data to tensors
u_data = np.array(u_data)
y_data = np.array(y_data)

u_data = torch.FloatTensor(u_data).transpose(0,1)
#u_data = u_data[0,:1000].unsqueeze(0).transpose(0,1)
y_data = torch.FloatTensor(y_data).transpose(0,1)
#y_data = y_data[0,:1000].unsqueeze(0).transpose(0,1)
#y_data = y_data[0,:].unsqueeze(1)
#torch.reshape(y_data,(1000, 1))
input_dim = list(u_data.shape)[1]
output_dim = list(y_data.shape)[1]

# Hyper parameters
T_ini = 5
T = y_data.shape[0]
N =10
DataShuffle = False
norm_data = True


u_data_train = u_data[0:round(0.8*T)]
y_data_train = y_data[0:round(0.8*T)]
U_ini, Y_ini,U_0_Nm1, Y_1_N = HankelMatrices(u_data_train,y_data_train,T_ini,N,round(0.8*T) , input_dim, output_dim )

trainX= torch.cat((U_ini, Y_ini, U_0_Nm1), 1)
trainy = Y_1_N


u_data_test = u_data
y_data_test = y_data
U_ini, Y_ini,U_0_Nm1, Y_1_N = HankelMatrices(u_data_test,y_data_test,T_ini,N,round(T) , input_dim, output_dim )

testX = torch.cat((U_ini, Y_ini, U_0_Nm1), 1)
testy = Y_1_N

Hankel Matrices Generated
Hankel Matrices Generated


In [7]:
class Net(nn.Module):
    def __init__(self,in_dim,out_dim):
        super().__init__()
        self.config = MambaConfig(d_model=8, n_layers=1)
        self.lin1 = nn.Linear(in_dim,8)
        self.mamba = Mamba(self.config)
        self.lin2 = nn.Linear(8,out_dim)
        
    
    def forward(self,x):
        #print("Input Shape:"+str(x.shape))
        x = self.lin1(x.unsqueeze(0))
        x = self.mamba(x)
        x = self.lin2(x)
        return x.squeeze(0)
        

In [8]:
model = Net(19,10)
model.load_state_dict(torch.load("State_dicts/Test_model_Conv1E1"))
model.eval()


C:\Users\Michiel\AppData\Local\Temp\ipykernel_61144\1617569546.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("State_dicts/Test_model_C

Net(
  (lin1): Linear(in_features=19, out_features=8, bias=True)
  (mamba): Mamba(
    (layers): ModuleList(
      (0): ResidualBlock(
        (mixer): MambaBlock(
          (in_proj): Linear(in_features=8, out_features=16, bias=False)
          (conv1d): Conv1d(8, 8, kernel_size=(1,), stride=(1,), groups=8)
          (x_proj): Linear(in_features=8, out_features=17, bias=False)
          (dt_proj): Linear(in_features=1, out_features=8, bias=True)
          (out_proj): Linear(in_features=8, out_features=8, bias=False)
        )
        (norm): RMSNorm()
      )
    )
    (norm_f): RMSNorm()
  )
  (lin2): Linear(in_features=8, out_features=10, bias=True)
)

In [9]:
import types
def imports():
    for name, val in globals().items():
        if isinstance(val, types.ModuleType):
            yield val.__name__
list(imports())

['builtins',
 'builtins',
 'numpy',
 'casadi',
 'matplotlib.pyplot',
 'torch',
 'torch.nn',
 'torch.nn.functional',
 'argparse',
 'scipy',
 'l4casadi',
 'casadi',
 'types']

In [10]:
from acados_template import AcadosOcp, AcadosOcpSolver
from pendulum_model import export_pendulum_ode_model
import numpy as np
import casadi as ca
from utils import plot_pendulum

def main():
    # create ocp object to formulate the OCP
    ocp = AcadosOcp()

    # set model
    model = export_pendulum_ode_model()
    ocp.model = model

    Tf = 1.0
    nx = model.x.rows()
    nu = model.u.rows()
    N = 20

    # set prediction horizon
    ocp.solver_options.N_horizon = N
    ocp.solver_options.tf = Tf

    # cost matrices
    Q_mat = 2*np.diag([1e3, 1e3, 1e-2, 1e-2])
    R_mat = 2*np.diag([1e-2])

    # path cost
    ocp.cost.cost_type = 'NONLINEAR_LS'
    ocp.model.cost_y_expr = ca.vertcat(model.x, model.u)
    ocp.cost.yref = np.zeros((nx+nu,))
    ocp.cost.W = ca.diagcat(Q_mat, R_mat).full()

    # terminal cost
    ocp.cost.cost_type_e = 'NONLINEAR_LS'
    ocp.cost.yref_e = np.zeros((nx,))
    ocp.model.cost_y_expr_e = model.x
    ocp.cost.W_e = Q_mat

    # set constraints
    Fmax = 80
    ocp.constraints.lbu = np.array([-Fmax])
    ocp.constraints.ubu = np.array([+Fmax])
    ocp.constraints.idxbu = np.array([0])

    ocp.constraints.x0 = np.array([0.0, np.pi, 0.0, 0.0])

    # set options
    ocp.solver_options.qp_solver = 'PARTIAL_CONDENSING_HPIPM' # FULL_CONDENSING_QPOASES
    # PARTIAL_CONDENSING_HPIPM, FULL_CONDENSING_QPOASES, FULL_CONDENSING_HPIPM,
    # PARTIAL_CONDENSING_QPDUNES, PARTIAL_CONDENSING_OSQP, FULL_CONDENSING_DAQP
    ocp.solver_options.hessian_approx = 'GAUSS_NEWTON' # 'GAUSS_NEWTON', 'EXACT'
    ocp.solver_options.integrator_type = 'IRK'
    # ocp.solver_options.print_level = 1
    ocp.solver_options.nlp_solver_type = 'SQP' # SQP_RTI, SQP
    ocp.solver_options.globalization = 'MERIT_BACKTRACKING' # turns on globalization

    ocp_solver = AcadosOcpSolver(ocp)

    simX = np.zeros((N+1, nx))
    simU = np.zeros((N, nu))

    status = ocp_solver.solve()
    ocp_solver.print_statistics() # encapsulates: stat = ocp_solver.get_stats("statistics")

    if status != 0:
        raise Exception(f'acados returned status {status}.')

    # get solution
    for i in range(N):
        simX[i,:] = ocp_solver.get(i, "x")
        simU[i,:] = ocp_solver.get(i, "u")
    simX[N,:] = ocp_solver.get(N, "x")

    plot_pendulum(np.linspace(0, Tf, N+1), Fmax, simU, simX, latexify=True, time_label=model.t_label, x_labels=model.x_labels, u_labels=model.u_labels)


if __name__ == '__main__':
    main()

Please export ACADOS_SOURCE_DIR to avoid this warning.
Please export ACADOS_SOURCE_DIR to avoid this warning.
Please export ACADOS_SOURCE_DIR to avoid this warning.
Please export ACADOS_SOURCE_DIR to avoid this warning.
Please export ACADOS_SOURCE_DIR to avoid this warning.
Please export ACADOS_SOURCE_DIR to avoid this warning.
Please export ACADOS_SOURCE_DIR to avoid this warning.
Please export ACADOS_SOURCE_DIR to avoid this warning.
Please export ACADOS_SOURCE_DIR to avoid this warning.
Please export ACADOS_SOURCE_DIR to avoid this warning.
Please export ACADOS_SOURCE_DIR to avoid this warning.
Please export ACADOS_SOURCE_DIR to avoid this warning.
Please export ACADOS_SOURCE_DIR to avoid this warning.
Please export ACADOS_SOURCE_DIR to avoid this warning.
Please export ACADOS_SOURCE_DIR to avoid this warning.
Please export ACADOS_SOURCE_DIR to avoid this warning.
Please export ACADOS_SOURCE_DIR to avoid this warning.
Please export ACADOS_SOURCE_DIR to avoid this warning.
Please exp

FileNotFoundError: [WinError 2] The system cannot find the file specified

In [24]:
model.load_state_dict(torch.load("State_dicts/Test_model_Conv1E1"))
model.eval()
MambaModel = l4c.L4CasADi(model, device='cpu')
MambaModel(testX[1,:].unsqueeze(0))

C:\Users\Michiel\AppData\Local\Temp\ipykernel_1560\3928445713.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("State_dicts/Test_model_Co

Exception: Compilation failed!

Attempted to execute OS command:
cmake -S _l4c_generated -B _l4c_generated -DCMAKE_RULE_MESSAGES=OFF && cmake --build _l4c_generated --config=Release&&  cmake --install _l4c_generated --config=Release



In [ ]:
#Sampling Time
Ts = 0.1

# Simulation time
time_range =24

# Simulation steps
L = round(time_range/Ts)

# Init time
time = np.arange(L+1)*Ts

# Initial state
x1_init = 0
x2_init = 0

x1 = np.zeros(L + 1)
x2 = np.zeros(L + 1)
u = np.zeros(L)
ref = np.zeros(L)

#Choose the reference
f = 0.2  # Frequency in Hz
ref =  np.sin(2 * np.pi * f * time)

# Init arrays to initial state
x1[0] = x1_init
x2[0] = x2_init

#Maximum input
u_max = 15

#Model Variables
N = 10
Tini =5

#Controller Variables
Q = 100
R =  0.5
Rd = 0
S = 100

#Casadi variables 
u = cs.MX.sym('u', N) 
y = cs.MX.sym('y', N)
ref = cs.MX.sym('ref', N)
u_init = cs.MX.sym('u_init') 
u_ini = cs.MX.sym('u_ini', Tini-1) 
y_ini = cs.MX.sym('y_ini', Tini)
# input = cs.vertcat(u_ini, y_ini,u)
inputs = cs.MX.sym('inputs', 19)
model_output = cs.MX.sym('model_output',10)
# gvar = cs.SX.sym('gvar', length(u_data)-N)
model = Net(19,10)
model.load_state_dict(torch.load("State_dicts/Test_model_Conv4E8"))
model.eval()
MambaModel = l4c.L4CasADi(model, device='cpu')


# Define cost function (example: quadratic tracking)
cost = 0
cost += (u[1]- u_init)*Rd*(u[1]- u_init)  # Control effort
cost +=  u[N-1]*R*u[N-1]
cost +=  (y[N-1] - ref[N-1])*S*(y[N-1] - ref[N-1])
for i in range(N-1):
    cost += (y[i] - ref[i])*Q*(y[i] - ref[i])  # Tracking error

for i in range(N-2):
    cost += (u[i+1]- u[i])*Rd*(u[i+1]- u[i])  # Control effort
    cost +=  u[i]*R*u[i]
# Create a CasADi function for the cost
f_cost = cs.Function('f_cost', [u, y, ref,u_init], [cost]) 

# Model Constraint
constraint1 =  MambaModel(inputs.T).T-y

# Define lower and upper bounds for the box constraint
lower_bound = -u_max  # Example lower bound
upper_bound = u_max  # Example upper boun

constraint2 = u <= upper_bound  # Upper bound constraint
constraint3 = u >= lower_bound  # Lower bound constraint
constraint4 = inputs- cs.vertcat(u_ini, y_ini,u)

# Create a CasADi function for the constraint
f_constraint = cs.Function('f_constraint', [u], [constraint1]) 

# Example usage within an optimization problem:
# ... (Your optimization problem setup) ...
nlp_prob = {'x': u, 'f': f_cost, 'g': f_constraint} 

# Define the solver parameters
opts = {}
opts['ipopt.print_level'] = 0  # Set to 0 for less output

# Create the NLP solver
solver = cs.nlpsol('solver', 'ipopt', nlp_prob, opts)


# Initial guess for the optimization variable (replace with your initial guess)
u0 = 1

# Solve the optimization problem
sol = solver(x0=u0)

# Extract the solution
x_opt = sol['x'] 

print("Optimal solution:", x_opt)


# # ---------- SIMULATION ----------

# for idx in range(1):

#     # ---------- SIMULATE USER INPUT ----------
#     if idx == 0:
#         y_ini = np.concatenate((y_ini[1:], [x1[idx]])) 
#         u_ini = u_ini 

#     elif idx >= 1:
#         y_ini = np.concatenate((y_ini[1:], [x1[idx]])) 
#         u_ini = np.concatenate((u_ini[1:], [uvec[idx-1]])) 
    
#     # Increment time
#     time += Ts

#     # ---------- CONTROL SYSTEM LOOP ----------


C:\Users\Michiel\AppData\Local\Temp\ipykernel_61144\1628341689.py:55: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("State_dicts/Test_model_

Exception: Compilation failed!

Attempted to execute OS command:
cmake -S _l4c_generated -B _l4c_generated -DCMAKE_RULE_MESSAGES=OFF && cmake --build _l4c_generated --config=Release&&  cmake --install _l4c_generated --config=Release



In [16]:
MambaModel = l4c.L4CasADi(model, device='cpu')
MambaModel(cs.vertcat(u_ini,y_ini,u).T)-y


c:\Users\Michiel\anaconda3\envs\run_controller\Lib\site-packages\torch\jit\_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(


Exception: Compilation failed!

Attempted to execute OS command:
cmake -S _l4c_generated -B _l4c_generated -DCMAKE_RULE_MESSAGES=OFF && cmake --build _l4c_generated --config=Release&&  cmake --install _l4c_generated --config=Release



In [21]:
model = Net(19,10)
model.load_state_dict(torch.load("State_dicts/Test_model_Conv1E1"))
model.eval()
MambaModel = l4c.L4CasADi(model, device='cpu')


input= np.random.randn(1,19)
MambaModel(input)

C:\Users\Michiel\AppData\Local\Temp\ipykernel_61144\1120899836.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("State_dicts/Test_model_C

Exception: Compilation failed!

Attempted to execute OS command:
cmake -S _l4c_generated -B _l4c_generated -DCMAKE_RULE_MESSAGES=OFF && cmake --build _l4c_generated --config=Release&&  cmake --install _l4c_generated --config=Release



In [9]:
def mpc_cost(variables,ref, Ts,mu,u_init, Q, R,Rd,S, N):

     # Initialise cost = 0 and states to current measured states
     cost = 0
     y, u = variables
     # Simulate the next N steps of the system
     for idx in range(N):
          if idx==1:
          # Update cost
               cost += (y[idx]-ref[idx])*Q*(y[idx]-ref[idx]) + u[idx]*R*u[idx] + (u[idx]-u_init)*Rd*(u[idx]-u_init)
          else:
               cost += (y[idx]-ref[idx])*Q*(y[idx]-ref[idx]) + u[idx]*R*u[idx] + (u[idx]-u[idx-1])*Rd*(u[idx]-[idx-1])
          
     return cost

In [3]:
# ---------- SIMULATION INITIALISATION ----------

Ts = 0.1

# Simulation time
time_range =24

# Simulation steps
L = round(time_range/Ts)

# Init time
time = np.arange(L+1)*Ts

# Initial state
x1_init = 0
x2_init = 0

x1 = np.zeros(L + 1)
x2 = np.zeros(L + 1)
u = np.zeros(L)
ref = np.zeros(L)

# Init arrays to initial state
x1[0] = x1_init
x2[0] = x2_init

#Choose the reference
f = 0.2  # Frequency in Hz
ref =  np.sin(2 * np.pi * f * time)

# ---------- CONTROL SYSTEM CALIBRATION ----------

# Model predictive control horizon
N = 10

Ts = 0.1
# Cost weights
Q = 100
R = 0.5 # control input ignored - constraints are in place
Rd = 0
S = 100

# Max torque allowed
u_max = 15

# Plant parameters - estimated
mu = 1  # VDP parameter

# Array for initial solution for MPC
u_ini = np.zeros(N)
y_ini = np.zeros(N)
uvec = np.zeros(L+1)
u_init = 0

# ---------- SIMULATION ----------

for idx in range(1):

    # ---------- SIMULATE USER INPUT ----------
    if idx == 0:
        y_ini = np.concatenate((y_ini[1:], [x1[idx]])) 
        u_ini = u_ini 

    elif idx >= 1:
        y_ini = np.concatenate((y_ini[1:], [x1[idx]])) 
        u_ini = np.concatenate((u_ini[1:], [uvec[idx-1]])) 
    
    # Increment time
    time += Ts

    # ---------- CONTROL SYSTEM LOOP ----------
    
    # Find optimal sequence for the next N steps
    u_mpc = solve_mpc(ref[idx], Ts,mu, Q, R,Rd,S, N, u_max,u_ini,y_ini,u_init )
    
    # Use first element of control input optimal solution
    uvec[idx] = u_mpc[0]

    # Initial solution for next step = current solution
    u_init = u_mpc

    # ---------- SIMULATION LOOP  ----------

    # Run dynamic system - pendulum
    (x1[idx+1], x2[idx+1]) = VDP(x1,x2, Ts,mu,u_mpc[0] )

# Plot results

plt.subplot(2, 1, 1)
plt.plot(np.arange(L+1)*Ts, x1[:], label="Theta [deg]")
#plt.plot(np.arange(L+1)*Ts, x2[:], label="Angular speed [deg/s]")
plt.plot(np.arange(L)*Ts, ref, '--', label="Setpoint [deg]")
plt.xlabel("Time")
plt.legend()
plt.grid()

plt.subplot(2, 1, 2)
plt.plot(np.arange(L)*Ts, u, label="Tau [Nm]")
plt.xlabel("Time")
plt.legend()
plt.grid()
plt.show()

NameError: name 'solve_mpc' is not defined